
# HealthConnect Rwanda - Week 6 Data Science
## Healthcare Predictive Analytics and Population Health Intelligence
**Intern:** Takudzwa Jeffrey Gumbo (TJ)  
**Programme:** AnalystLab Africa Experience Lab - Batch E

### Responsible-use statement
This project uses the healthcare datasets supplied for the AnalystLab Africa Week 6 assignment. All risk indicators, predictions and forecasts are analytical demonstrations only. They are not clinical diagnoses, treatment recommendations, or a substitute for qualified healthcare professionals. The supplied Health Facilities master table was not available; facility-level outputs therefore use anonymised FacilityIDs and do not invent names, ownership or geographic mappings.


In [1]:

from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt, warnings
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import *
from sklearn.inspection import permutation_importance
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from statsmodels.tsa.holtwinters import Holt
warnings.filterwarnings('ignore')

if Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')
    SEARCH_ROOT=Path('/content/drive/MyDrive/Analyst Lab Africa')
else:
    SEARCH_ROOT=Path('/mnt/data')

def find_file(name):
    direct=SEARCH_ROOT/name
    if direct.exists(): return direct
    matches=list(SEARCH_ROOT.rglob(name))
    if not matches: raise FileNotFoundError(f'Missing {name} below {SEARCH_ROOT}')
    return matches[0]
print('Search root:',SEARCH_ROOT)


Mounted at /content/drive
Search root: /content/drive/MyDrive/Analyst Lab Africa


## 1. Data loading and relationship audit

In [2]:

csv_files=['HCCountries.csv', 'HC_Provinces.csv', 'HC_Districts.csv', 'HC_Sectors.csv', 'HC_Villages.csv', 'HC_Ownership_Types.csv', 'HC_Departments.csv', 'HC_Medical_Specialties.csv', 'HC_Services.csv', 'HC_Medical_Equipment.csv', 'HC_Rooms.csv', 'HC_Beds.csv', 'HC_Doctor.csv', 'HC_Nurse.csv', 'Pharmacist.csv', 'Laboratory_Scientist.csv', 'HC_Radiographer.csv', 'HC_Administrative_Staff.csv', 'HC_Patients.csv', 'HC_Diagnoses.csv', 'HC_Billing.csv', 'Insurance_Claims.csv', 'HC_Payments.csv', 'HC_Patient_Feedback.csv', 'HC_Revenue_Summary.csv']
paths={f:find_file(f) for f in csv_files}
tables={Path(f).stem:pd.read_csv(p) for f,p in paths.items()}
clinical_path=find_file('HC_Clinical_Transactions.xlsx')
xl=pd.ExcelFile(clinical_path)
clinical={s:pd.read_excel(clinical_path,sheet_name=s) for s in xl.sheet_names}
summary=[]
for name,df in list(tables.items())+[(f'Clinical_{k}',v) for k,v in clinical.items()]:
    summary.append([name,len(df),df.shape[1],int(df.isna().sum().sum()),int(df.duplicated().sum())])
display(pd.DataFrame(summary,columns=['Dataset','Rows','Columns','MissingCells','DuplicateRows']))


,Dataset,Rows,Columns,MissingCells,DuplicateRows
0,HCCountries,1,6,0,0
1,HC_Provinces,5,3,0,0
2,HC_Districts,30,3,0,0
3,HC_Sectors,120,3,0,0
4,HC_Villages,480,3,0,0
5,HC_Ownership_Types,4,2,0,0
6,HC_Departments,200,3,0,0
7,HC_Medical_Specialties,200,3,0,0
8,HC_Services,400,3,0,0
9,HC_Medical_Equipment,200,4,0,0


### Data limitation identified before modelling
The detailed `Appointments` sheet contains 5,000 appointment records, while full diagnoses/billing tables extend to `APT060000` and the enterprise monthly summary reports 60,809 appointments. Therefore, appointment-level predictive modelling is restricted to the 5,000 detailed appointments; enterprise forecasting uses the 12-month summary. The Health Facilities master table is also absent. DepartmentID-FacilityID pairings are highly inconsistent in appointments and staff relative to the department master, so DepartmentID is used for department labels but the supplied department-facility hierarchy is not treated as reliable for cross-facility joins.


## 2. Cleaning, validation and feature engineering

In [3]:

patients=tables['HC_Patients'].copy(); patients['DateOfBirth']=pd.to_datetime(patients['DateOfBirth'],errors='coerce')
appointments=clinical['Appointments'].copy(); appointments['Date']=pd.to_datetime(appointments['Date'],errors='coerce')
diagnoses=tables['HC_Diagnoses']; billing=tables['HC_Billing']; feedback=tables['HC_Patient_Feedback']
admissions=clinical['Admissions']; labs=clinical['Laboratory_Tests']; prescriptions=clinical['Prescriptions']; referrals=clinical['Referrals']
departments=tables['HC_Departments']

a=appointments.merge(patients,on='PatientID',how='left',validate='many_to_one',suffixes=('','_Patient'))
a=a.merge(diagnoses[['AppointmentID','Diagnosis','Severity']],on='AppointmentID',how='left',validate='one_to_one')
a=a.merge(billing[['AppointmentID','BillingID','Amount','Status']].rename(columns={'Status':'BillingStatus'}),on='AppointmentID',how='left',validate='one_to_one')
a['Admitted']=a['AppointmentID'].isin(set(admissions.AppointmentID)).astype(int)
a=a.merge(admissions[['AppointmentID','LengthOfStayDays']],on='AppointmentID',how='left')
a['HasLab']=a['AppointmentID'].isin(set(labs.AppointmentID)).astype(int)
a['HasPrescription']=a['AppointmentID'].isin(set(prescriptions.AppointmentID)).astype(int)
a['HasReferral']=a['AppointmentID'].isin(set(referrals.AppointmentID)).astype(int)
a=a.merge(feedback[['AppointmentID','Rating','Comment']],on='AppointmentID',how='left')
a=a.merge(departments[['DepartmentID','DepartmentName']],on='DepartmentID',how='left')
a['AgeGroup']=pd.cut(a.Age,[-1,17,34,49,64,200],labels=['0-17','18-34','35-49','50-64','65+'])
a['HasChronicCondition']=a.ChronicCondition.notna().astype(int)
a['BMI_Category']=pd.cut(a.BMI,[-np.inf,18.5,25,30,np.inf],labels=['Underweight','Normal','Overweight','Obese'],right=False)
a['SeverityScore']=a.Severity.map({'Mild':1,'Moderate':2,'Severe':3})
a['SatisfactionCategory']=pd.cut(a.Rating,[0,2,3,5],labels=['Low','Medium','High'])
a['RevenueCategory']=pd.qcut(a.Amount,4,labels=['Low','Lower-Mid','Upper-Mid','High'])
a=a.sort_values(['PatientID','Date','AppointmentID'])
a['PreviousAppointments']=a.groupby('PatientID').cumcount()
a['FrequentVisitorIndicator']=(a.groupby('PatientID').AppointmentID.transform('count')>=2).astype(int)
a['HighRiskPatientIndicator']=((a.Age>=65)|(a.HasChronicCondition==1)|(a.BMI>=30)).astype(int)
print('Integrated detailed sample:',a.shape)
print('Admission rate:',a.Admitted.mean())
print('Matched satisfaction records:',a.Rating.notna().sum())


Integrated detailed sample: (5000, 47)
Admission rate: 0.1172
Matched satisfaction records: 1756


## 3. EDA and statistical analysis

In [4]:

print('Patients:',len(patients))
print('Mean age:',patients.Age.mean())
print('Recorded chronic-condition prevalence:',patients.ChronicCondition.notna().mean())
print('Mean satisfaction (all feedback):',feedback.Rating.mean())
print('Total billing amount:',billing.Amount.sum())


Patients: 20000
Mean age: 45.0332
Recorded chronic-condition prevalence: 0.74645
Mean satisfaction (all feedback): 2.9871819217622484
Total billing amount: 9309401961


In [5]:

# H1 satisfaction across facilities
sat_groups=[g.Rating.dropna().values for _,g in a[a.Rating.notna()].groupby('FacilityID') if len(g)>=5]
t1=stats.kruskal(*sat_groups)
# H2 age group vs admission
ct=pd.crosstab(a.AgeGroup,a.Admitted); t2=stats.chi2_contingency(ct)
# H3 billing distribution by chronic-condition status
t3=stats.mannwhitneyu(a.loc[a.HasChronicCondition==1,'Amount'],a.loc[a.HasChronicCondition==0,'Amount'],alternative='two-sided')
print('H1 p-value:',t1.pvalue)
print('H2 p-value:',t2.pvalue)
print('H3 p-value:',t3.pvalue)


H1 p-value: 0.7952304278874007
H2 p-value: 0.03075610600425485
H3 p-value: 0.5438271047832024


## 4. Forecasting

In [6]:

revenue=tables['HC_Revenue_Summary'].copy(); revenue['Month']=pd.to_datetime(revenue.Month+'-01'); revenue=revenue.sort_values('Month')
forecasts={}
for target in ['Appointments','Revenue']:
    model=Holt(revenue[target].astype(float).values,damped_trend=True,initialization_method='estimated').fit(optimized=True)
    forecasts[target]=model.forecast(12)
future=pd.date_range(revenue.Month.max()+pd.offsets.MonthBegin(1),periods=12,freq='MS')
display(pd.DataFrame({'Month':future,'AppointmentsForecast':forecasts['Appointments'],'RevenueForecast':forecasts['Revenue']}))


,Month,AppointmentsForecast,RevenueForecast
0,2027-01-01,4662.019910,9.856648e+07
1,2027-02-01,4644.570428,9.912627e+07
2,2027-03-01,4630.610842,9.964979e+07
3,2027-04-01,4619.443173,1.001394e+08
4,2027-05-01,4610.509038,1.005973e+08
5,2027-06-01,4603.361730,1.010255e+08
6,2027-07-01,4597.643884,1.014260e+08
7,2027-08-01,4593.069607,1.018005e+08
8,2027-09-01,4589.410185,1.021508e+08
9,2027-10-01,4586.482647,1.024784e+08


## 5. Hospital admission prediction

In [7]:

features=['Age','Gender','MaritalStatus','Education','BloodGroup','BMI','HasChronicCondition','VisitType','FacilityID','DepartmentID','Diagnosis','Severity']
X=a[features]; y=a.Admitted
num=['Age','BMI','HasChronicCondition']; cat=[c for c in features if c not in num]
pre=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore'))]),cat)])
mods={'Logistic Regression':LogisticRegression(max_iter=2000,class_weight='balanced',random_state=42),'Random Forest':RandomForestClassifier(n_estimators=250,max_depth=10,min_samples_leaf=5,class_weight='balanced',random_state=42,n_jobs=-1),'Gradient Boosting':GradientBoostingClassifier(random_state=42)}
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
rows=[]
for name,m in mods.items():
    p=Pipeline([('pre',pre),('model',m)]); p.fit(Xtr,ytr); prob=p.predict_proba(Xte)[:,1]; pred=(prob>=.5).astype(int)
    rows.append([name,roc_auc_score(yte,prob),accuracy_score(yte,pred),precision_score(yte,pred,zero_division=0),recall_score(yte,pred,zero_division=0),f1_score(yte,pred,zero_division=0)])
display(pd.DataFrame(rows,columns=['Model','ROC-AUC','Accuracy','Precision','Recall','F1']).sort_values('ROC-AUC',ascending=False))


,Model,ROC-AUC,Accuracy,Precision,Recall,F1
2,Gradient Boosting,0.496723,0.881,0.000000,0.000000,0.000000
0,Logistic Regression,0.495939,0.532,0.109131,0.418803,0.173145
1,Random Forest,0.482853,0.728,0.102564,0.170940,0.128205


## 6. Patient satisfaction prediction

In [8]:

sat=a[a.Rating.notna()].copy(); features=['Age','Gender','BMI','HasChronicCondition','VisitType','FacilityID','DepartmentID','Diagnosis','Severity','Amount','Admitted','HasLab','HasPrescription','HasReferral']
X=sat[features]; y=sat.Rating.astype(float)
num=['Age','BMI','HasChronicCondition','Amount','Admitted','HasLab','HasPrescription','HasReferral']; cat=[c for c in features if c not in num]
pre=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore'))]),cat)])
mods={'Linear Regression':LinearRegression(),'Random Forest':RandomForestRegressor(n_estimators=250,max_depth=10,min_samples_leaf=5,random_state=42,n_jobs=-1),'Gradient Boosting':GradientBoostingRegressor(random_state=42,max_depth=2,n_estimators=120)}
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42)
rows=[]
for name,m in mods.items():
    p=Pipeline([('pre',pre),('model',m)]); p.fit(Xtr,ytr); pred=p.predict(Xte); rows.append([name,mean_absolute_error(yte,pred),mean_squared_error(yte,pred)**0.5,r2_score(yte,pred)])
display(pd.DataFrame(rows,columns=['Model','MAE','RMSE','R2']).sort_values('RMSE'))


,Model,MAE,RMSE,R2
1,Random Forest,1.212055,1.409000,-0.021546
2,Gradient Boosting,1.221659,1.418204,-0.034937
0,Linear Regression,1.277406,1.497929,-0.154565


## 7. Patient clustering

In [9]:

pagg=a.groupby('PatientID').agg(Visits=('AppointmentID','size'),MeanBilling=('Amount','mean'),Admissions=('Admitted','sum'),MeanRating=('Rating','mean')).reset_index()
cl=patients[['PatientID','Age','BMI','ChronicCondition']].copy(); cl['HasChronicCondition']=cl.ChronicCondition.notna().astype(int); cl=cl.merge(pagg,on='PatientID',how='left')
for c in ['Visits','MeanBilling','Admissions','MeanRating']: cl[c]=cl[c].fillna(0)
cols=['Age','BMI','HasChronicCondition','Visits','MeanBilling','Admissions','MeanRating']; X=StandardScaler().fit_transform(cl[cols]);
best=None
for k in [3,4,5]:
    km=KMeans(n_clusters=k,random_state=42,n_init=10).fit(X); idx=np.random.RandomState(42).choice(len(cl),min(5000,len(cl)),replace=False); s=silhouette_score(X[idx],km.labels_[idx]);
    if best is None or s>best[0]: best=(s,k,km)
cl['Cluster']=best[2].labels_; print('Selected k:',best[1],'silhouette:',best[0]); display(cl.groupby('Cluster')[cols].mean().round(2))


Selected k: 3 silhouette: 0.43867126411247964


,Age,BMI,HasChronicCondition,Visits,MeanBilling,Admissions,MeanRating
Cluster,,,,,,,
0,44.87,29.99,0.75,0.02,464.24,0.00,0.00
1,47.21,29.65,0.76,1.25,155311.97,1.01,1.17
2,45.39,30.33,0.74,1.11,163596.13,0.00,1.18


## 8. Conclusions and responsible use
The supplied project data supports descriptive analysis and demonstration models, but several predictive tasks show weak generalisation. That result is retained rather than tuned away. Facility names/ownership/geography cannot be linked because the Health Facilities master was not supplied, and DepartmentID-FacilityID pairs are inconsistent across several tables. Forecasts are based on only 12 monthly enterprise observations and should be treated as planning illustrations, not operational commitments. The high-risk indicator is an educational heuristic only and is not a clinical risk score.
